# Spark UI Deep Dive: Forensic Analysis of Jobs, Stages, and Tasks

**Module**: 10_performance_and_optimization  
**Notebook**: 01_spark_ui_analysis.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Mastering the Spark UI: Jobs, Stages, Storage, Environment, Executors, and SQL tabs, event timelines, and diagnostic triage.
- **Why does this topic exist?**: The Spark UI is your primary window into distributed execution. 90% of tuning decisions originate from UI metrics.
- **What problem does it solve?**: Trying to optimize jobs blindly without knowing if CPU, network, disk, or GC is the bottleneck.
- **Where is it used in real-world Data Engineering?**: Resolving enterprise cluster SLA breaches and performance incidents.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Jobs/stages/tasks, shuffle, memory basics |
| **Required depth** | Working knowledge: can navigate the UI and form a evidence-based performance hypothesis |
| **Data Engineer** | Use the Spark UI to locate the bottleneck before changing code or configuration. |
| **Senior Data Engineer** | Correlate stage metrics with data distribution and cluster behavior. |
| **Spark specialist** | Reason about scheduler delay, GC, spill, and skew from task-level metrics. |
| **Next topic** | `10_performance_and_optimization/02_explain_and_execution_plans.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The Spark UI is the primary diagnostic instrument. It shows which stage dominates, how tasks are distributed,
how much data is shuffled or spilled, and whether the cluster is busy or waiting. Without it, tuning is guesswork;
with it, a vague "job is slow" becomes a specific stage, task, and metric to investigate.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Jobs tab** | Actions and their stages | One job per action, with stage progress |
| **Stages tab** | Stage metrics | Task duration, shuffle read/write, spill, input/output |
| **Tasks tab** | Task-level detail | Duration percentiles, scheduler delay, GC time |
| **SQL tab** | Query plans and metrics | Physical plan with per-operator statistics |
| **Storage tab** | Cached blocks | Cached size, fraction, evictions |
| **Executors tab** | Resource usage | Cores, memory, GC, shuffle service |

### How it works, step by step

1. Open the application in the History Server or live UI.
2. Start at the Jobs tab and find the slow action.
3. Open the stage with the longest duration or largest shuffle.
4. Inspect task duration percentiles and the max/median ratio.
5. Check shuffle read/write, spill, GC, and input size.
6. Correlate with the SQL plan to identify the operator.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Navigating the Spark UI
- **Jobs Tab**: High-level timeline of triggered actions.
- **Stages Tab**: Deeper metrics per stage: Duration percentiles (Min, 25%, Median, 75%, Max), Shuffle Read/Write bytes, Spill (Memory/Disk).
- **Storage Tab**: Cached DataFrames/RDDs, memory used, percentage cached in RAM.
- **Executors Tab**: Active cores, memory used per executor, GC pause time, failed tasks.
- **SQL Tab**: Visual DAG plan with physical operators and row counters.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Spark_UI_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
res.show()
print("Open http://localhost:4040/ to inspect the live Spark UI!")

---

## Experiment: Correlating Spark UI Event Timeline with Code

- **1. What are we testing?**: Run a multi-stage join query and inspect Event Timeline bars.
- **2. What do we expect?**: Timeline shows blue (compute), orange (fetch wait), and red (GC).

In [ ]:
products = spark.read.option("header", True).csv("/opt/spark-data/retail/products.csv")
orders.join(products, "product_id").groupBy("category").count().show()
print("Check http://localhost:4040/stages/ for Stage task timelines!")

**3. What happened?**: Spark UI tracked execution timeline across all executor cores.  
**4. Why did it happen?**: LiveListenerBus streams task events to the embedded web server.  
**5. Practical Engineering Lesson**: Large gaps or long bars in the timeline pinpoint the exact execution bottleneck.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Nightly SLA breach

- **Industry**: Retail
- **Scenario**: A nightly revenue job misses its SLA by two hours.
- **Problem**: The team guesses that memory is the problem and requests bigger executors.
- **Why this concept is relevant**: The UI shows one stage with a max task duration far above the median, indicating skew rather than memory pressure.
- **Implementation idea**: Inspect the task distribution, identify the hot key, and apply a skew remediation.
- **Expected result**: A targeted fix that restores the SLA without adding cluster capacity.
- **Alternative**: Add more executors.
- **Trade-offs**: More executors help only if there is enough parallel work; skew leaves them idle.

### Use Case 2: Cost review

- **Industry**: SaaS
- **Scenario**: A platform team must reduce Spark spend by 20%.
- **Problem**: Jobs are over-provisioned because no one knows which stages actually use the resources.
- **Why this concept is relevant**: UI metrics show low CPU utilization and large shuffle writes, pointing to partition and join changes rather than larger clusters.
- **Implementation idea**: Profile the top jobs, rank stages by shuffle and CPU, and optimize the largest contributors first.
- **Expected result**: A prioritized optimization backlog with measured savings.
- **Alternative**: Move everything to a managed service.
- **Trade-offs**: A managed service may reduce operational cost but does not automatically fix inefficient jobs.

---

## Business Value

- Faster diagnosis and fewer wasted tuning cycles.
- Evidence-based capacity decisions.
- Lower compute cost from targeted fixes.
- A shared performance vocabulary across teams.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Always start with the stage that dominates wall-clock time.
- Compare max and median task duration to detect skew.
- Check spill and GC alongside shuffle metrics.
- Use the SQL tab to connect metrics to operators.
- Save event logs for post-mortems and regression comparisons.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Find the dominant stage

**Steps**

1. Run a multi-stage job with a join and an aggregation.
2. Open the Jobs tab and identify the longest job.
3. Open its stages and record duration, shuffle, spill, and max/median task duration.

**What to observe**: One stage usually dominates; its metrics point to the next diagnostic step.

**Healthy result**: You can name the bottleneck and the metric that supports it.

**Problematic result**: All stages look similar, suggesting the bottleneck is outside Spark or spread across the pipeline.

**Engineering takeaway**: Triage from the largest cost, not from the most familiar setting.

---

## Trade-offs and Alternatives

### Advantages

- Built into Spark
- Task-level detail
- Historical event logs

### Limitations

- Can be overwhelming
- Live UI disappears when the app ends
- Metrics need interpretation

### When to use it

- Performance triage
- Capacity reviews
- Regression analysis

### When not to use it

- Guessing without metrics
- Looking only at job duration
- Ignoring task percentiles

### Alternatives inside Spark

- Spark UI
- History Server
- SQL tab
- Event logs

### Alternatives outside Spark

- APM tools
- Cluster monitoring
- Query profilers

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check UI Port
**Problem Statement**: What port hosts the driver UI?  
**Input / Context**: Port

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Identify Straggler Task
**Problem Statement**: How to spot stragglers in Stages tab.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Executor GC Ratio
**Problem Statement**: Calculate GC percentage threshold.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The stage that looks fine

**Scenario**: A stage has a short average task duration but the job still takes hours.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the max task duration and the task-duration distribution.
2. Look for a few tasks with huge shuffle read.
3. Inspect skew and partition sizes.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check UI Port):
```python
print('Port 4040 (Driver) and Port 18080 (History Server)')
```

#### Solution 2 (Identify Straggler Task):
```python
print('Compare Median vs Max task duration. If Max >> Median, it is a straggler.')
```

#### Solution 3 (Executor GC Ratio):
```python
print('If Task GC time exceeds 10% of total duration, investigate JVM memory allocation.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **UI Forensic Flow**: Jobs -> Stages -> Task Summary Table -> SQL Plan.
- **Straggler Detection**: Max task time >> 75th percentile.

### Essential Technical Terms
- `LiveListenerBus`: Internal event queue feeding UI.
- `Event Timeline`: Visual timeline of task threads.

### Common Engineering Mistakes
- **Closing browser and losing metrics**: Use History Server (18080) for completed jobs.

### Interview Questions on This Topic
- **Q: What does an orange segment in the Spark UI Task Timeline indicate?**
  - *A*: Shuffle Fetch Wait time (waiting for remote shuffle map output blocks over the network).


### More Interview Questions

- **Q: Where do you start when a Spark job is slow?**
  - *A*: With the Spark UI: identify the job and stage that dominate wall-clock time, then inspect task duration distribution, shuffle, spill, and GC.
- **Q: What does a large max-to-median task duration ratio indicate?**
  - *A*: Skew or an uneven partition: a few tasks process much more data than the median, so the stage waits for the slowest task.

### What to Study Next
Open 02_explain_and_execution_plans.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')